In [ ]:
#[Problem 1] Code review


Differences from the last used implementation:

 **Data Augmentation:** The previous implementation might have included more extensive data augmentation techniques. This notebook primarily focuses on resizing and adding depth/absolute depth channels.
 
**Model Architecture:** This notebook specifically uses a U-Net architecture with a ResNet50 encoder for transfer learning. The previous implementation might have used a different model architecture.
Loss Functions and Metrics: This notebook explores several loss functions (Dice, BCE, combined, Lovasz) and a custom IoU metric. Your previous implementation might have used a different set of loss functions and evaluation metrics.

**Threshold Optimization:** This notebook includes a step for optimizing the prediction threshold based on the IoU metric, which might not have been explicitly done in the previous implementation.

**How Transfer Learning is done:**

Transfer learning is implemented by using the ResNet50 model, pretrained on the ImageNet dataset, as the encoder part of the U-Net architecture.

**1.Loading the Pretrained Model:** The ResNet50 model is loaded with weights='imagenet', which means it comes with the weights learned from the ImageNet dataset. The include_top=False argument is crucial as it removes the classification layers at the top of the network, allowing it to be used as a feature extractor.

**2.Feature Extraction:** The code extracts features from intermediate layers of the pretrained ResNet50 model (conv1, res2c_branch2c, res3d_branch2c, res4f_branch2c, res5c_branch2c). These layers represent different levels of abstraction in the image features.

**3.Building the Decoder:** Custom decoder blocks (decoder_block_simple or decoder_block_bottleneck) are built to upsample the features extracted by the encoder.

**4.Skip Connections:** Skip connections are implemented by concatenating the upsampled output of each decoder block with the corresponding feature map from the encoder. This allows the model to combine high-level semantic information from the decoder with low-level spatial information from the encoder, which is essential for accurate segmentation.

**5.Training:** The entire U-Net model, with the pretrained ResNet50 encoder, is then trained on your specific dataset (salt segmentation). During training, the weights of the ResNet50 encoder can either be frozen (not updated) or fine-tuned along with the decoder weights. This notebook appears to fine-tune the entire model.

In essence, the pretrained ResNet50 provides a strong starting point for feature extraction, which helps the U-Net model learn to segment the salt masks more effectively, especially with a limited dataset.

In [ ]:
#[Problem 2] Code rewriting

In [ ]:
# [Problem 3] Learning / estimation

In [ ]:
## The code for the modified unet_resnet function with VGG16 encoder


In [3]:
import tensorflow as tf
from keras.losses import binary_crossentropy
from keras import backend as K
from keras.applications.vgg16 import VGG16
from keras.layers import Conv2D, BatchNormalization, PReLU, Dropout, Add, concatenate, UpSampling2D, Activation, Input
from keras.models import Model

# Basic decoder block with Conv, BN and PReLU activation.
def decoder_block_simple(
        layer_name, block_name,
        num_filters=32,
        conv_dim=(3, 3)):

    x_dec = Conv2D(
        num_filters, conv_dim,
        padding='same',
        name='{}_conv'.format(block_name))(layer_name)
    x_dec = BatchNormalization(
        name='{}_bn'.format(block_name))(x_dec)
    x_dec = PReLU(
        name='{}_activation'.format(block_name))(x_dec)

    return x_dec

# Decoder block with bottleneck architecture, where middle conv layer
# is half the size of first and last, in order to compress representation.
# This type of architecture is supposed to retain most useful information.
def decoder_block_bottleneck(
        layer_name, block_name,
        num_filters=32,
        conv_dim=(3, 3),
        dropout_frac=0.2):

    x_dec = Conv2D(
        num_filters, conv_dim,
        padding='same',
        name='{}_conv1'.format(block_name))(layer_name)
    x_dec = BatchNormalization(
        name='{}_bn1'.format(block_name))(x_dec)
    x_dec = PReLU(
        name='{}_activation1'.format(block_name))(x_dec)
    x_dec = Dropout(dropout_frac)(x_dec)

    x_dec2 = Conv2D(
        num_filters // 2, conv_dim,
        padding='same',
        name='{}_conv2'.format(block_name))(x_dec)
    x_dec2 = BatchNormalization(
        name='{}_bn2'.format(block_name))(x_dec2)
    x_dec2 = PReLU(
        name='{}_activation2'.format(block_name))(x_dec2)
    x_dec2 = Dropout(dropout_frac)(x_dec2)

    x_dec2 = Conv2D(
        num_filters, conv_dim,
        padding='same',
        name='{}_conv3'.format(block_name))(x_dec2)
    x_dec2 = BatchNormalization(
        name='{}_bn3'.format(block_name))(x_dec2)
    x_dec2 = PReLU(
        name='{}_activation3'.format(block_name))(x_dec2)
    x_dec2 = Dropout(dropout_frac)(x_dec2)

    x_dec2 = Add()([x_dec, x_dec2])

    return x_dec2


# IoU metric for observation during training
# https://www.kaggle.com/cpmpml/fast-iou-metric-in-numpy-and-tensorflow
def get_iou_vector(A, B):
    # Numpy version
    batch_size = A.shape[0]
    metric = 0.0
    for batch in range(batch_size):
        t, p = A[batch], B[batch]
        true = np.sum(t)
        pred = np.sum(p)

        # deal with empty mask first
        if true == 0:
            metric += (pred == 0)
            continue

        # non empty mask case.  Union is never empty
        # hence it is safe to divide by its number of pixels
        intersection = np.sum(t * p)
        union = true + pred - intersection
        iou = intersection / union

        # iou metrric is a stepwise approximation of the real iou over 0.5
        iou = np.floor(max(0, (iou - 0.45)*20)) / 10

        metric += iou

    # teake the average over all images in batch
    metric /= batch_size
    return metric


def my_iou_metric(label, pred):
    return tf.py_func(get_iou_vector, [label, pred>0.5], tf.float64)


# For Lovash loss
def my_iou_metric_2(label, pred):
    return tf.py_func(get_iou_vector, [label, pred >0], tf.float64)

# Model is parametrized in a way to enable easy change of decoder_block type,
# as this is an argument that can be given a function, like decoder_block_simple.
def unet_resnet(input_size, decoder_block,
                weights='imagenet',
                loss_func='binary_crossentropy',
                metrics_list=[my_iou_metric],
                use_lovash=False):

    # Base model - encoder (using VGG16)
    base_model = VGG16(
        input_shape=input_size,
        include_top=False,
        weights=weights)

    # Layers for feature extraction in the encoder part (VGG16)
    # These are chosen to match the spatial resolutions of the original ResNet50 layers
    encoder1 = base_model.get_layer('block1_conv2').output # shape: (None, 224, 224, 64)
    encoder2 = base_model.get_layer('block2_conv2').output # shape: (None, 112, 112, 128)
    encoder3 = base_model.get_layer('block3_conv3').output # shape: (None, 56, 56, 256)
    encoder4 = base_model.get_layer('block4_conv3').output # shape: (None, 28, 28, 512)
    encoder5 = base_model.get_layer('block5_conv3').output # shape: (None, 14, 14, 512)


    # Center block
    center = decoder_block(
        encoder5, 'center', num_filters=512)
    concat5 = concatenate([center, encoder5], axis=-1)

    # Decoder part.
    # Every decoder block processed concatenated output from encoder and decoder part.
    # This creates skip connections.
    # Afterwards, decoder output is upsampled to dimensions equal to encoder output part.
    decoder4 = decoder_block(
        concat5, 'decoder4', num_filters=256)
    concat4 = concatenate([UpSampling2D()(decoder4), encoder4], axis=-1)

    decoder3 = decoder_block(
        concat4, 'decoder3', num_filters=128)
    concat3 = concatenate([UpSampling2D()(decoder3), encoder3], axis=-1)

    decoder2 = decoder_block(
        concat3, 'decoder2', num_filters=64)
    concat2 = concatenate([UpSampling2D()(decoder2), encoder2], axis=-1)

    decoder1 = decoder_block(
        concat2, 'decoder1', num_filters=64)
    concat1 = concatenate([UpSampling2D()(decoder1), encoder1], axis=-1)

    # Final upsampling and decoder block for segmentation.
    output = UpSampling2D()(concat1)
    output = decoder_block(
        output, 'decoder_output', num_filters=32)
    output = Conv2D(
        1, (1, 1), activation=None, name='prediction')(output)
    if not use_lovash:
        output = Activation('sigmoid')(output)

    model = Model(base_model.input, output)
    model.compile(loss=loss_func, optimizer='adam', metrics=metrics_list)

    return model

In [ ]:
#[Problem 3] Learning / estimation

### Expected Differences: ResNet vs. VGG

| Feature | VGG ( VGG16) | ResNet ( ResNet50) |
| :--- | :--- | :--- |
| **Key Architectural Feature** | Simple stack of $3 \times 3$ convolutions and max-pooling. | **Residual (Skip) Connections**. |
| **Depth/Parameters** | Less deep, but **more parameters** (due to many large fully-connected layers, though often removed for segmentation). | **Much deeper**, but **fewer parameters** (due to $1 \times 1$ bottleneck and no FC layers). |
| **Gradient Flow** | Can suffer from the vanishing gradient problem in very deep architectures. | **Improved gradient flow** due to skip connections, which helps training very deep networks. |
| **Typical Performance** | Often converges faster. Can be effective for segmentation, especially with the U-Net architecture. | Usually achieves **higher overall accuracy** in classification and often in segmentation if fully leveraged. |
